# Discipline Classifier v4.0 - Advanced Pipeline
# Target: 95%+ Accuracy with LoRA, SciNCL, Data Augmentation, and Ensemble

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install transformers==4.41.0 accelerate==0.27.0 peft==0.8.2 datasets==2.16.0 -q
!pip install scikit-learn xgboost nlpaug -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2023.10.0 which is incompatible.


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda


In [3]:
# Load your dataset
data_path = "/content/drive/MyDrive/NLP_Project/expanded_discipline_with_preds.csv"
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Use existing columns
df['text'] = df['text_input']
df['label'] = df['v2.2_predicted_label']
df['trust_score'] = df['v2.2_trust_score']

# Label mapping
id2label = {0: 'CS', 1: 'IS', 2: 'IT'}
label2id = {'CS': 0, 'IS': 1, 'IT': 2}

print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

Dataset shape: (5402, 7)
Columns: ['Title', 'Abstract', 'Discipline', 'Link', 'text_input', 'v2.2_predicted_label', 'v2.2_trust_score']

Label distribution:
label
0    3037
1    1644
2     721
Name: count, dtype: int64


### IT Class Data Augmentation

In [6]:
import nlpaug.augmenter.word as naw
import random

class SimpleAugmenter:
    def __init__(self):
        # Use multiple augmentation strategies
        self.synonym_aug = naw.SynonymAug(aug_src='wordnet', aug_p=0.3)
        self.random_aug = naw.RandomWordAug(action='swap')

    def augment_text_simple(self, text):
        """Simple augmentation by swapping words and adding synonyms"""
        # Method 1: Random word swap
        words = text.split()
        if len(words) > 10:
            # Swap 2-3 random word pairs
            for _ in range(random.randint(2, 3)):
                idx1, idx2 = random.sample(range(len(words)), 2)
                words[idx1], words[idx2] = words[idx2], words[idx1]
            return ' '.join(words)
        return text

    def augment_minority_classes(self, df):
        """Focus on augmenting IT class which has only 721 samples"""
        augmented_rows = []

        # Count current distribution
        current_dist = df['label'].value_counts().sort_index()
        print(f"Current distribution: {current_dist.to_dict()}")

        # IT class (label=2) needs the most help
        it_df = df[df['label'] == 2].copy()
        is_df = df[df['label'] == 1].copy()

        # Target: bring IT to ~2000 samples (need ~1300 more)
        n_augment_it = 2000 - len(it_df)
        n_augment_is = 2000 - len(is_df)

        print(f"\nAugmentation plan:")
        print(f"IT: {len(it_df)} -> {len(it_df) + n_augment_it}")
        print(f"IS: {len(is_df)} -> {len(is_df) + n_augment_is}")

        # Augment IT samples
        print(f"Starting IT augmentation (this may take a few minutes)...")
        for i in range(min(n_augment_it, 1300)):  # Cap at 1300 to avoid too long processing
            if i % 50 == 0:  # More frequent updates
                print(f"  Generated {i}/{n_augment_it} IT augmentations...")

            row = it_df.sample(1).iloc[0].copy()
            # Try synonym augmentation first
            try:
                aug_text = self.synonym_aug.augment(row['text'])[0] if isinstance(self.synonym_aug.augment(row['text']), list) else self.synonym_aug.augment(row['text'])
            except:
                # Fallback to simple augmentation
                aug_text = self.augment_text_simple(row['text'])

            if aug_text != row['text']:  # Only add if actually changed
                new_row = row.copy()
                new_row['text'] = aug_text
                new_row['trust_score'] = row['trust_score'] * 0.9
                new_row['is_augmented'] = True
                augmented_rows.append(new_row)

        print(f"  Completed IT augmentation: {len([r for r in augmented_rows if r['label'] == 2])} samples created")

        # Augment IS samples (less aggressively)
        print(f"Starting IS augmentation...")
        initial_aug_count = len(augmented_rows)
        for i in range(min(n_augment_is, 400)):  # Smaller number for IS
            if i % 50 == 0:  # More frequent updates
                print(f"  Generated {i}/{n_augment_is} IS augmentations...")

            row = is_df.sample(1).iloc[0].copy()
            try:
                aug_text = self.random_aug.augment(row['text'])[0] if isinstance(self.random_aug.augment(row['text']), list) else self.random_aug.augment(row['text'])
            except:
                aug_text = self.augment_text_simple(row['text'])

            if aug_text != row['text']:
                new_row = row.copy()
                new_row['text'] = aug_text
                new_row['trust_score'] = row['trust_score'] * 0.9
                new_row['is_augmented'] = True
                augmented_rows.append(new_row)

        print(f"  Completed IS augmentation: {len(augmented_rows) - initial_aug_count} samples created")

        print(f"\nTotal augmented samples: {len(augmented_rows)}")

        # Combine
        if augmented_rows:
            aug_df = pd.DataFrame(augmented_rows)
            df['is_augmented'] = False  # Mark original samples
            df_combined = pd.concat([df, aug_df], ignore_index=True)
            return df_combined

        print("WARNING: No augmentations were created!")
        return df


In [7]:
# Apply augmentation
print("\nAugmenting minority classes...")
augmenter = SimpleAugmenter()
df_augmented = augmenter.augment_minority_classes(df)

print(f"\nAugmented dataset shape: {df_augmented.shape}")
print("New label distribution:")
print(df_augmented['label'].value_counts().sort_index())

# Verify augmentation worked
if 'is_augmented' in df_augmented.columns:
    print(f"\nAugmented samples: {df_augmented['is_augmented'].sum()}")
else:
    print("\nWARNING: Augmentation may have failed!")


Augmenting minority classes...
Current distribution: {0: 3037, 1: 1644, 2: 721}

Augmentation plan:
IT: 721 -> 2000
IS: 1644 -> 2000
Starting IT augmentation (this may take a few minutes)...
  Generated 0/1279 IT augmentations...
  Generated 50/1279 IT augmentations...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

  Generated 100/1279 IT augmentations...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

  Generated 150/1279 IT augmentations...


...

  Generated 1100/1279 IT augmentations...
  Generated 1150/1279 IT augmentations...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

  Generated 1200/1279 IT augmentations...
  Generated 1250/1279 IT augmentations...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

  Completed IT augmentation: 1279 samples created
Starting IS augmentation...
  Generated 0/356 IS augmentations...
  Generated 50/356 IS augmentations...
  Generated 100/356 IS augmentations...
  Generated 150/356 IS augmentations...
  Generated 200/356 IS augmentations...
  Generated 250/356 IS augmentations...
  Generated 300/356 IS augmentations...
  Generated 350/356 IS augmentations...
  Completed IS augmentation: 356 samples created

Total augmented samples: 1635

Augmented dataset shape: (7037, 11)
New label distribution:
label
0    3037
1    2000
2    2000
Name: count, dtype: int64

Augmented samples: 1635


### Focal Loss

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss

        return focal_loss.mean()

### Load Model with LoRA (if PEFT works)

In [9]:
try:
    from peft import LoraConfig, get_peft_model, TaskType
    USE_LORA = True
    print("PEFT available - will use LoRA")
except:
    USE_LORA = False
    print("PEFT not available - using standard fine-tuning")

# Load model
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if USE_LORA:
    # Create base model
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    # Apply LoRA
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "value"]
    )
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()
else:
    # Standard model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

PEFT available - will use LoRA


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 297,219 || all params: 110,217,990 || trainable%: 0.2696646890403282


### Prepare Data

In [10]:
# Split data
X = df_augmented['text'].values
y = df_augmented['label'].values
trust_scores = df_augmented['trust_score'].values

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nClass weights: {class_weights}")

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Create datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = Dataset.from_dict({"text": X_train, "labels": y_train})
val_dataset = Dataset.from_dict({"text": X_val, "labels": y_val})

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])


Class weights: tensor([0.7724, 1.1728, 1.1728], device='cuda:0')


Map:   0%|          | 0/5629 [00:00<?, ? examples/s]

Map:   0%|          | 0/1408 [00:00<?, ? examples/s]

### Custom trainer with focal loss

In [11]:
class FocalLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=gamma, alpha=class_weights)

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [13]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    learning_rate=2e-5,
    save_total_limit=2,
    report_to="none",
    seed=42
)

In [14]:
# Compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_per_class = f1_score(labels, predictions, average=None)

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_cs': f1_per_class[0],
        'f1_is': f1_per_class[1],
        'f1_it': f1_per_class[2]
    }

In [15]:
# Create trainer
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

In [16]:
# Train
print("\n" + "="*50)
print("Starting training with focal loss...")
print("="*50)

trainer.train()


Starting training with focal loss...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Cs,F1 Is,F1 It
200,0.483600,0.432924,0.586648,0.548394,0.708843,0.473856,0.462485
400,0.294800,0.225235,0.816761,0.807092,0.855510,0.845336,0.720430
600,0.166400,0.151716,0.850142,0.845754,0.878814,0.859447,0.799001
800,0.127000,0.123060,0.873580,0.868484,0.900166,0.878860,0.826425
1000,0.124300,0.114022,0.879972,0.874694,0.908638,0.876485,0.838961
1200,0.117600,0.107079,0.889915,0.885518,0.913580,0.886199,0.856774
1400,0.100000,0.103314,0.889205,0.884883,0.914238,0.881437,0.858974
1600,0.094500,0.101656,0.897017,0.893216,0.918249,0.892944,0.868455


TrainOutput(global_step=1760, training_loss=0.19329188140955839, metrics={'train_runtime': 748.9222, 'train_samples_per_second': 37.581, 'train_steps_per_second': 2.35, 'total_flos': 7431025124689920.0, 'train_loss': 0.19329188140955839, 'epoch': 5.0})

In [17]:
# Evaluate
print("\nEvaluating model...")
eval_results = trainer.evaluate()

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

# Detailed classification report
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['CS', 'IS', 'IT']))


Evaluating model...



FINAL RESULTS
eval_loss: 0.1017
eval_accuracy: 0.8970
eval_f1_macro: 0.8932
eval_f1_cs: 0.9182
eval_f1_is: 0.8929
eval_f1_it: 0.8685
eval_runtime: 13.2466
eval_samples_per_second: 106.2910
eval_steps_per_second: 3.3220
epoch: 5.0000

Classification Report:
              precision    recall  f1-score   support

          CS       0.92      0.91      0.92       608
          IS       0.87      0.92      0.89       400
          IT       0.89      0.85      0.87       400

    accuracy                           0.90      1408
   macro avg       0.89      0.89      0.89      1408
weighted avg       0.90      0.90      0.90      1408



In [18]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"/content/drive/MyDrive/NLP_Project/discipline_classifier_v4_{timestamp}"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"\nModel saved to: {save_path}")


Model saved to: /content/drive/MyDrive/NLP_Project/discipline_classifier_v4_20250610_051005


### Ensemble Model

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb

print("\nTraining ensemble model...")

# Extract TF-IDF features
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# Train XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softprob'
)
xgb_model.fit(X_train_tfidf, y_train)

# Evaluate XGBoost alone
xgb_pred = xgb_model.predict(X_val_tfidf)
xgb_acc = accuracy_score(y_val, xgb_pred)
xgb_f1 = f1_score(y_val, xgb_pred, average='macro')

print(f"\nXGBoost Performance:")
print(f"Accuracy: {xgb_acc:.4f}")
print(f"Macro F1: {xgb_f1:.4f}")



Training ensemble model...

XGBoost Performance:
Accuracy: 0.9276
Macro F1: 0.9263


In [20]:
# Save ensemble components
import joblib
joblib.dump(tfidf, f"{save_path}/tfidf_vectorizer_v4.0.pkl")
joblib.dump(xgb_model, f"{save_path}/xgb_model_v4.0.pkl")

['/content/drive/MyDrive/NLP_Project/discipline_classifier_v4_20250610_051005/xgb_model_v4.0.pkl']

In [21]:
print("\n✅ Training complete!")
print(f"Expected performance: 85-90% accuracy")
print("\nNext steps for 95%:")
print("1. Use SciNCL model: model_name = 'malteos/scincl'")
print("2. Add more IT training data (GPT-4 generation)")
print("3. Multi-task learning with subfields")


✅ Training complete!
Expected performance: 85-90% accuracy

Next steps for 95%:
1. Use SciNCL model: model_name = 'malteos/scincl'
2. Add more IT training data (GPT-4 generation)
3. Multi-task learning with subfields
